# 🎬 Netflix Content Recommender System
**Data Analyst Internship Project**

**Objective:** Build a content-based recommendation engine on the Netflix Titles dataset.  
**Dataset:** [Netflix Movies and TV Shows — Kaggle](https://www.kaggle.com/datasets/shivamb/netflix-shows)  
**Algorithm:** TF-IDF Vectorisation + Cosine Similarity

---
### Table of Contents
1. [Setup & Imports](#1)
2. [Data Collection](#2)
3. [Data Understanding](#3)
4. [Data Cleaning & Preprocessing](#4)
5. [Exploratory Data Analysis (EDA)](#5)
6. [Feature Engineering](#6)
7. [Recommendation Engine](#7)
8. [Demo & Results](#8)
9. [Conclusion](#9)

## 1. Setup & Imports <a id='1'></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import LabelEncoder

# Plot style
plt.rcParams.update({
    'figure.facecolor': '#141414',
    'axes.facecolor':   '#1f1f1f',
    'axes.edgecolor':   '#333333',
    'text.color':       '#F5F5F1',
    'axes.labelcolor':  '#888888',
    'xtick.color':      '#888888',
    'ytick.color':      '#888888',
    'grid.color':       '#333333',
})
RED = '#E50914'
print('✅ Imports complete')

## 2. Data Collection <a id='2'></a>

In [ ]:
df = pd.read_csv('data/netflix_titles.csv')
print(f'Shape: {df.shape}')
df.head()

## 3. Data Understanding <a id='3'></a>

In [ ]:
df.info()

In [ ]:
print('Columns:', df.columns.tolist())
print('\nNull counts:\n', df.isnull().sum())

In [ ]:
df.describe(include='all')

## 4. Data Cleaning & Preprocessing <a id='4'></a>

In [ ]:
# Fill missing values
df['director'] = df['director'].fillna('Unknown')
df['cast']     = df['cast'].fillna('Unknown')
df['country']  = df['country'].fillna('Unknown')
df['rating']   = df['rating'].fillna('Not Rated')

# Drop duplicates
df.drop_duplicates(inplace=True)

# Clean title
df['title'] = df['title'].str.strip()

# Parse dates
df['date_added'] = pd.to_datetime(df['date_added'], errors='coerce')
df['year_added'] = df['date_added'].dt.year

print(f'Clean shape: {df.shape}')
print('Remaining nulls:\n', df.isnull().sum())

## 5. Exploratory Data Analysis <a id='5'></a>

In [ ]:
# Movies vs TV Shows
fig, ax = plt.subplots(figsize=(7,5))
counts = df['type'].value_counts()
ax.bar(counts.index, counts.values, color=[RED, '#F5F5F1'], width=0.5)
ax.set_title('Movies vs TV Shows', fontsize=14)
plt.show()

In [ ]:
# Top 15 genres
genres = df['listed_in'].str.split(', ').explode()
top_genres = genres.value_counts().head(15)
fig, ax = plt.subplots(figsize=(12,6))
top_genres.plot(kind='bar', ax=ax, color=RED, edgecolor='none')
ax.set_title('Top 15 Genres', fontsize=14)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Content added over time
year_counts = df['year_added'].dropna().astype(int).value_counts().sort_index()
fig, ax = plt.subplots(figsize=(12,5))
ax.fill_between(year_counts.index, year_counts.values, alpha=0.3, color=RED)
ax.plot(year_counts.index, year_counts.values, color=RED, lw=2.5, marker='o', ms=5)
ax.set_title('Content Added Per Year', fontsize=14)
plt.show()

In [ ]:
# Release year histogram
fig, ax = plt.subplots(figsize=(12,5))
sns.histplot(df['release_year'].dropna(), bins=30, color=RED, edgecolor='none', ax=ax)
ax.set_title('Release Year Distribution', fontsize=14)
plt.show()

In [ ]:
# Rating distribution
fig, ax = plt.subplots(figsize=(10,6))
order = df['rating'].value_counts().index
sns.countplot(y='rating', data=df, order=order, color=RED, ax=ax)
ax.set_title('Rating Distribution', fontsize=14)
plt.show()

## 6. Feature Engineering <a id='6'></a>

In [ ]:
# Combine text features
df['combined_features'] = (
    df['listed_in'].fillna('') + ' ' +
    df['description'].fillna('') + ' ' +
    df['director'].fillna('') + ' ' +
    df['cast'].fillna('')
)
df['combined_features'].head(3)

In [ ]:
# TF-IDF Vectorisation
tfidf = TfidfVectorizer(stop_words='english', max_features=10000, ngram_range=(1,2))
tfidf_matrix = tfidf.fit_transform(df['combined_features'])
print('TF-IDF matrix shape:', tfidf_matrix.shape)

In [ ]:
# Cosine similarity
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)
print('Cosine similarity matrix shape:', cosine_sim.shape)

## 7. Recommendation Engine <a id='7'></a>

In [ ]:
# Build title → index map
indices = pd.Series(df.index, index=df['title']).drop_duplicates()

def recommend(title, top_n=10):
    """Return top_n most similar Netflix titles."""
    if title not in indices.index:
        close = [t for t in indices.index if title.lower() in t.lower()][:3]
        return f"'{title}' not found. Did you mean: {close}"
    
    idx    = indices[title]
    scores = sorted(enumerate(cosine_sim[idx]), key=lambda x: x[1], reverse=True)
    scores = [(i,s) for i,s in scores if i != idx][:top_n]
    
    rec_idx = [i for i,_ in scores]
    sims    = [round(s,4) for _,s in scores]
    
    result = df[['title','type','listed_in','rating']].iloc[rec_idx].copy()
    result['similarity'] = sims
    result.index = range(1, len(result)+1)
    return result

print('✅ Recommendation function ready')

## 8. Demo & Results <a id='8'></a>

In [ ]:
print('🎬 Recommendations for "Stranger Things":')
recommend('Stranger Things')

In [ ]:
print('🎬 Recommendations for "The Irishman":')
recommend('The Irishman')

## 9. Conclusion <a id='9'></a>

### Summary
- Loaded and cleaned **8,800+ Netflix titles** from Kaggle
- Conducted thorough EDA revealing content trends, genre distribution, and growth patterns
- Engineered a combined text feature from genre, description, director, and cast
- Built a **TF-IDF + cosine similarity** content-based recommender returning top-10 results

### Key Findings
- **~70% Movies**, ~30% TV Shows
- **Dramas** dominate the genre landscape
- Netflix content additions **peaked in 2019–2020**
- **TV-MA** is the most common rating
- **US content** is most prevalent, followed by India and UK

### Future Work
- Collaborative filtering using user watch history
- Sentence-BERT embeddings for richer semantic similarity
- Streamlit web app deployment
- Hybrid model combining content + collaborative signals